# PF200 ORB ML Research V1

Objetivo: construir un clasificador futures-native para puntuar candidatos ORB1/ORB2 sobre MNQ/MES sin reemplazar el motor deterministico.

Filosofia:
- train in Research
- persist bundle to ObjectStore
- main.py solo consume score/confidence
- promotion only if IS/OOS/STRESS improve without breaches


In [ ]:
import warnings, json, math, pickle, zlib, base64
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from QuantConnect.Research import QuantBook
from QuantConnect import Resolution
from QuantConnect.Algorithm import QCAlgorithm
from QuantConnect import Futures

try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except Exception:
    HAVE_LGBM = False

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
qb = QuantBook()
print('LGBM available:', HAVE_LGBM)


In [ ]:
CONFIG = {
    'start': datetime(2020, 1, 1),
    'end': datetime(2026, 3, 31),
    'or_minutes': 15,
    'slots': [('ORB1', 10, 15), ('ORB2', 11, 0)],
    'fwd_minutes': 120,
    'target_r': 1.0,
    'stop_atr_mult': 0.75,
    'target_atr_mult': 1.55,
    'symbols': ['MNQ','MES'],
    'objectstore_prefix': 'pf200_orb_ml_v1/'
}
CONFIG


In [ ]:
future_map = {
    'MNQ': Futures.Indices.MicroNASDAQ100EMini,
    'MES': Futures.Indices.MicroSP500EMini,
}
contracts = {}
for name, future_type in future_map.items():
    fut = qb.AddFuture(future_type, Resolution.Minute)
    fut.SetFilter(0, 182)
    contracts[name] = fut.Symbol
contracts


## Dataset builder
Construye ejemplos diarios por slot con features compatibles con el main actual: gap, OR width, ATR proxy, momentum, trend alignment, break quality y contexto de sesion.


In [ ]:
def _flatten_history(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.reset_index()
    cols = [c.lower() for c in out.columns]
    out.columns = cols
    return out

def fetch_histories(symbol, start, end):
    min_df = _flatten_history(qb.History(symbol, start, end, Resolution.Minute))
    day_df = _flatten_history(qb.History(symbol, start - timedelta(days=90), end, Resolution.Daily))
    return min_df, day_df

def atr14(day_df):
    d = day_df.copy()
    d['prev_close'] = d['close'].shift(1)
    tr = pd.concat([
        (d['high'] - d['low']).abs(),
        (d['high'] - d['prev_close']).abs(),
        (d['low'] - d['prev_close']).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(14).mean()

def build_dataset_for_symbol(name, symbol, cfg):
    min_df, day_df = fetch_histories(symbol, cfg['start'], cfg['end'])
    if min_df.empty or day_df.empty:
        return pd.DataFrame()
    min_df['time'] = pd.to_datetime(min_df['time'])
    min_df['date'] = min_df['time'].dt.date
    day_df['time'] = pd.to_datetime(day_df['time'])
    day_df['date'] = day_df['time'].dt.date
    day_df['atr14'] = atr14(day_df)
    day_df['trend55'] = day_df['close'].ewm(span=55, adjust=False).mean()
    daily = day_df[['date','open','high','low','close','atr14','trend55']].copy()
    rows = []
    for d, intraday in min_df.groupby('date'):
        if len(intraday) < 100:
            continue
        ref = daily[daily['date'] < d].tail(1)
        cur = daily[daily['date'] == d].tail(1)
        if ref.empty or cur.empty:
            continue
        prev_close = float(ref.iloc[0]['close'])
        atr = float(cur.iloc[0]['atr14']) if pd.notna(cur.iloc[0]['atr14']) else np.nan
        trend = float(cur.iloc[0]['trend55']) if pd.notna(cur.iloc[0]['trend55']) else np.nan
        if not np.isfinite(prev_close) or not np.isfinite(atr) or atr <= 0:
            continue
        open_bar = intraday.iloc[0]
        session_open = float(open_bar['open'])
        gap_pct = (session_open - prev_close) / prev_close
        or_end = pd.Timestamp(datetime.combine(pd.Timestamp(d).to_pydatetime().date(), datetime.min.time()) + timedelta(hours=9, minutes=30 + cfg['or_minutes']))
        or_bars = intraday[intraday['time'] <= or_end]
        if len(or_bars) < 3:
            continue
        or_high = float(or_bars['high'].max())
        or_low = float(or_bars['low'].min())
        or_width = or_high - or_low
        or_width_atr = or_width / atr
        for slot_name, hh, mm in cfg['slots']:
            slot_ts = pd.Timestamp(datetime.combine(pd.Timestamp(d).to_pydatetime().date(), datetime.min.time()) + timedelta(hours=hh, minutes=mm))
            now_bars = intraday[intraday['time'] <= slot_ts]
            if now_bars.empty:
                continue
            px = float(now_bars.iloc[-1]['close'])
            day_high = max(float(now_bars['high'].max()), px)
            day_low = min(float(now_bars['low'].min()), px)
            day_range_atr = (day_high - day_low) / atr
            intraday_mom = (px - session_open) / session_open if session_open > 0 else 0.0
            long_break = float(px > or_high)
            short_break = float(px < or_low)
            uptrend = float(px > trend) if np.isfinite(trend) else 0.0
            downtrend = float(px < trend) if np.isfinite(trend) else 0.0
            future_bars = intraday[(intraday['time'] > slot_ts) & (intraday['time'] <= slot_ts + timedelta(minutes=cfg['fwd_minutes']))]
            if future_bars.empty:
                continue
            long_max = float(future_bars['high'].max()) - px
            long_min = px - float(future_bars['low'].min())
            short_max = px - float(future_bars['low'].min())
            short_min = float(future_bars['high'].max()) - px
            stop_dist = atr * cfg['stop_atr_mult']
            target_dist = atr * cfg['target_atr_mult']
            y_long = int(long_max >= target_dist and long_min <= stop_dist) if stop_dist > 0 else 0
            y_short = int(short_max >= target_dist and short_min <= stop_dist) if stop_dist > 0 else 0
            rows.append({
                'symbol': name, 'date': str(d), 'slot': slot_name, 'px': px, 'prev_close': prev_close, 'session_open': session_open,
                'gap_pct': gap_pct, 'or_width_atr': or_width_atr, 'day_range_atr': day_range_atr, 'intraday_mom': intraday_mom,
                'long_break': long_break, 'short_break': short_break, 'uptrend': uptrend, 'downtrend': downtrend,
                'target_dist': target_dist, 'stop_dist': stop_dist, 'y_long': y_long, 'y_short': y_short
            })
    return pd.DataFrame(rows)


In [ ]:
frames = []
for name, symbol in contracts.items():
    df = build_dataset_for_symbol(name, symbol, CONFIG)
    print(name, df.shape)
    frames.append(df)
dataset = pd.concat([f for f in frames if not f.empty], ignore_index=True) if frames else pd.DataFrame()
dataset.head(), dataset.shape


In [ ]:
features = ['gap_pct','or_width_atr','day_range_atr','intraday_mom','long_break','short_break','uptrend','downtrend']
dataset['date'] = pd.to_datetime(dataset['date'])
is_mask = (dataset['date'] >= '2022-01-01') & (dataset['date'] <= '2024-12-31')
oos_mask = (dataset['date'] >= '2025-01-01') & (dataset['date'] <= '2026-03-31')
stress_mask = (dataset['date'] >= '2020-01-01') & (dataset['date'] <= '2020-12-31')
train = dataset[is_mask].copy()
oos = dataset[oos_mask].copy()
stress = dataset[stress_mask].copy()
target_col = 'y_long'
if HAVE_LGBM:
    model = LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
else:
    model = RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=20, random_state=42, n_jobs=-1)
pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', model)])
pipe.fit(train[features], train[target_col])
for name, frame in [('IS', train), ('OOS', oos), ('STRESS', stress)]:
    if frame.empty:
        print(name, 'empty')
        continue
    probs = pipe.predict_proba(frame[features])[:,1]
    pred = (probs >= 0.55).astype(int)
    auc = roc_auc_score(frame[target_col], probs) if frame[target_col].nunique() > 1 else np.nan
    prec = precision_score(frame[target_col], pred, zero_division=0)
    rate = float(pred.mean())
    print(name, {'auc': round(float(auc),4) if np.isfinite(auc) else None, 'precision': round(float(prec),4), 'signal_rate': round(rate,4), 'n': int(len(frame))})


In [ ]:
bundle = {
    'version': 'pf200_orb_ml_v1',
    'created_utc': datetime.utcnow().isoformat(),
    'features': features,
    'target_col': target_col,
    'threshold': 0.55,
    'config': CONFIG,
    'model_bytes_b64': base64.b64encode(zlib.compress(pickle.dumps(pipe, protocol=pickle.HIGHEST_PROTOCOL), 9)).decode('ascii'),
}
manifest = {
    'version': bundle['version'],
    'created_utc': bundle['created_utc'],
    'features': features,
    'threshold': bundle['threshold'],
    'rows': int(len(dataset)),
}
model_key = CONFIG['objectstore_prefix'] + 'bundle.json'
manifest_key = CONFIG['objectstore_prefix'] + 'manifest.json'
qb.ObjectStore.Save(model_key, json.dumps(bundle))
qb.ObjectStore.Save(manifest_key, json.dumps(manifest, indent=2))
print('saved:', model_key, manifest_key)
manifest
